## Establecer conexión con SQL

In [56]:
import mysql.connector
from mysql.connector import Error
import pandas as pd

In [57]:
try:
    connection = mysql.connector.connect(host='212.227.90.6',
                                         database='Equip_15',
                                         user='Equipo15',
                                         password = 'E1q2u3i4p5o15')
    if connection.is_connected():
        db_Info = connection.get_server_info()
        print("Connected to MySQL Server version ", db_Info)
        RRHH = pd.read_sql(f"SELECT * FROM RRHH_15092025", con=connection)
        
       
except Error as e:
    print("Error while connecting to MySQL", e)

Connected to MySQL Server version  8.0.43-0ubuntu0.24.04.1


C:\Users\gemma\AppData\Local\Temp\ipykernel_11508\859305398.py:7: DeprecationWarning: Call to deprecated function get_server_info. Reason: 
    The property counterpart 'server_info' should be used instead.

  db_Info = connection.get_server_info()
C:\Users\gemma\AppData\Local\Temp\ipykernel_11508\859305398.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  RRHH = pd.read_sql(f"SELECT * FROM RRHH_15092025", con=connection)


In [58]:
pd.set_option("display.max.columns", None)
#RRHH.describe() #variables numèriques
#RRHH.describe(include = 'object') #variables categòriques
RRHH.dtypes

ID                          int64
Reason_absence              int64
Month_absence               int64
Day_week                    int64
Seasons                     int64
Transportation_expense      int64
Distance_Residence_Work     int64
Service_time                int64
Age                         int64
Work_load_Average_day      object
Hit_target                  int64
Disciplinary_failure       object
Education                  object
Son                        object
Social_drinker             object
Social_smoker              object
Pet                        object
Weight                      int64
Height                      int64
Body_mass_index             int64
Absenteeism_hours           int64
dtype: object

# Limpieza de datos

## Cambiar los tipos de datos

In [59]:
# Change from numeric to categorical
for col in ["ID", "Reason_absence","Month_absence","Day_week","Seasons"]:
    RRHH[col] = RRHH[col].astype("object")
    
# Change from object to numeric
for col in ["Son", "Pet"]:
    RRHH[col] = RRHH[col].astype("int")

# Change decimal separator and change type to float
RRHH["Work_load_Average_day"] = (
    RRHH["Work_load_Average_day"]
    .str.replace(",", ".", regex=False) 
    .astype(float)
)

RRHH.dtypes

ID                          object
Reason_absence              object
Month_absence               object
Day_week                    object
Seasons                     object
Transportation_expense       int64
Distance_Residence_Work      int64
Service_time                 int64
Age                          int64
Work_load_Average_day      float64
Hit_target                   int64
Disciplinary_failure        object
Education                   object
Son                          int64
Social_drinker              object
Social_smoker               object
Pet                          int64
Weight                       int64
Height                       int64
Body_mass_index              int64
Absenteeism_hours            int64
dtype: object

In [60]:
RRHH.describe()
#RRHH.describe(include = 'object') 
#RRHH.describe(include = 'category') 

,Transportation_expense,Distance_Residence_Work,Service_time,Age,Work_load_Average_day,Hit_target,Son,Pet,Weight,Height,Body_mass_index,Absenteeism_hours
count,740.000000,740.000000,740.000000,740.000000,740.000000,740.000000,740.000000,740.000000,740.000000,740.000000,740.000000,740.000000
mean,221.329730,29.631081,12.554054,36.450000,271.490235,94.587838,1.018919,0.745946,79.035135,172.114865,26.677027,6.924324
std,66.952223,14.836788,4.384873,6.478772,39.058116,3.779313,1.098489,1.318258,12.883211,6.034995,4.285452,13.330998
min,118.000000,5.000000,1.000000,27.000000,205.917000,81.000000,0.000000,0.000000,56.000000,163.000000,19.000000,0.000000
25%,179.000000,16.000000,9.000000,31.000000,244.387000,93.000000,0.000000,0.000000,69.000000,169.000000,24.000000,2.000000
50%,225.000000,26.000000,13.000000,37.000000,264.249000,95.000000,1.000000,0.000000,83.000000,170.000000,25.000000,3.000000
75%,260.000000,50.000000,16.000000,40.000000,294.217000,97.000000,2.000000,1.000000,89.000000,172.000000,31.000000,8.000000
max,388.000000,52.000000,29.000000,58.000000,378.884000,100.000000,4.000000,8.000000,108.000000,196.000000,38.000000,120.000000


## Eliminar duplicados exactos

In [61]:
RRHH = RRHH.drop_duplicates()
RRHH.shape[0]

706

## Exportar a csv

In [62]:
RRHH.to_csv("RRHH_150925_clean.csv", index=False, sep=",")

## Cerrar conexión

In [63]:
connection.close()

# Transformación

## Renombrar niveles variables categóricas

In [64]:
# MESES
## Dejamos la columna con número para poderlo ordenar en PowerBI
RRHH["Month_absence_order"] = RRHH["Month_absence"]

# Especificamos el nombre del mes
month_map = {
    1: "Enero", 2: "Febrero", 3: "Marzo", 4: "Abril",
    5: "Mayo", 6: "Junio", 7: "Julio", 8: "Agosto",
    9: "Septiembre", 10: "Octubre", 11: "Noviembre", 12: "Diciembre"
}

RRHH["Month_absence"] = RRHH["Month_absence"].replace(month_map)


# DÍAS DE LA SEMANA
## Dejamos la columna con número para poderlo ordenar en PowerBI
RRHH["Day_week_order"] = RRHH["Day_week"]

# Especificamos el día de la semana
day_map = {
    2: "Lunes", 3: "Martes", 4: "Miercoles", 5: "Jueves", 6: "Viernes"
}
RRHH["Day_week"] = RRHH["Day_week"].replace(day_map)

# ESTACIONES DEL AÑO
spanish_season_map = {
    1: "Invierno", 2: "Otono", 3: "Verano", 4: "Primavera"
}
RRHH["Seasons"] = RRHH["Seasons"].replace(spanish_season_map)

# EDUCACIÓN
# Especificamos el nombre del mes
education_map = {
    1: "High school ", 2: "Graduate", 3: "Postgraduate", 4: "Master/Doctor"
}

RRHH["Education"] = RRHH["Education"].replace(month_map)

## Separamos la información en dos tablas 

### Hechos

In [65]:
RRHH_hechos = RRHH
RRHH_hechos.head()

,ID,Reason_absence,Month_absence,Day_week,Seasons,Transportation_expense,Distance_Residence_Work,Service_time,Age,Work_load_Average_day,Hit_target,Disciplinary_failure,Education,Son,Social_drinker,Social_smoker,Pet,Weight,Height,Body_mass_index,Absenteeism_hours,Month_absence_order,Day_week_order
0,14,11,Noviembre,Lunes,Primavera,155,12,14,34,284.031,97,0,1,2,1,0,0,95,196,25,120,11,2
1,36,13,Abril,Miercoles,Verano,118,13,18,50,239.409,98,0,1,1,1,0,0,98,178,31,120,4,4
2,9,6,Julio,Martes,Invierno,228,14,16,58,264.604,93,0,1,2,0,0,1,65,172,22,120,7,3
3,28,9,Julio,Martes,Invierno,225,26,9,28,230.290,92,0,1,1,0,0,2,69,169,24,112,7,3
4,9,12,Marzo,Martes,Otono,228,14,16,58,222.196,99,0,1,2,0,0,1,65,172,22,112,3,3


In [66]:
# Seleccionamos las columnas de interés

RRHH_hechos = RRHH[["ID", "Reason_absence", "Month_absence","Day_week","Seasons","Work_load_Average_day","Hit_target","Disciplinary_failure","Absenteeism_hours","Month_absence_order","Day_week_order"]]

### Dimensiones

In [67]:
RRHH_dimensiones = RRHH[["ID", "Transportation_expense", "Distance_Residence_Work","Service_time","Age","Education","Son","Social_drinker","Social_smoker","Pet","Weight","Height","Body_mass_index"]]

In [68]:
RRHH_dimensiones = RRHH_dimensiones.drop_duplicates(subset=["ID"], keep="last").reset_index(drop=True)
RRHH_dimensiones.shape[0]

36

In [69]:
RRHH_hechos.to_csv("RRHH_150925_hechos.csv", index=False, sep=",")
RRHH_dimensiones.to_csv("RRHH_150925_dimensiones.csv", index=False, sep=",")

# Clustering

In [70]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Agrupar por ids
socio = RRHH.groupby("ID").first()[["Age","Distance_Residence_Work","Transportation_expense" ,"Son", "Education", "Weight", "Height", "Pet", "Social_drinker", "Social_smoker"]]

# Adaptar las variables categoricas
categorical_cols = ["Education", "Social_drinker", "Social_smoker"]
for col in categorical_cols:
    socio[col] = socio[col].astype(str)
    socio[col] = socio[col].fillna("Desconocido")
df_socio = pd.get_dummies(socio, columns=categorical_cols, drop_first=True)

# Escalado
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_socio)

# Numero de clusters y resultados
kmeans = KMeans(n_clusters=4, random_state=42)
clusters = kmeans.fit_predict(X_scaled)
df_socio["cluster"] = clusters
cluster_summary = df_socio.groupby("cluster").mean()
print(cluster_summary)

# Ids de cada grupo
cluster_ids = {}
for c in sorted(df_socio["cluster"].unique()):
    ids = df_socio[df_socio["cluster"] == c].index.tolist()
    cluster_ids[c] = ids
    print(f"\nCluster {c} ({len(ids)} empleados):")
    print(ids)

               Age  Distance_Residence_Work  Transportation_expense       Son  \
cluster                                                                         
0        37.500000                13.500000              190.000000  2.000000   
1        41.454545                24.454545              262.818182  1.818182   
2        41.000000                34.230769              236.538462  0.923077   
3        31.600000                23.800000              219.600000  0.500000   

            Weight      Height       Pet  Education_2  Education_3  \
cluster                                                              
0        94.500000  189.000000  1.000000     0.000000          0.0   
1        71.545455  172.454545  1.000000     0.090909          0.0   
2        87.461538  170.846154  1.538462     0.000000          0.0   
3        72.900000  173.400000  1.300000     0.300000          0.3   

         Education_4  Social_drinker_1  Social_smoker_1  
cluster                           